*NLP @ HSE DSBA, autumn 2026. Author: Daria Andreeva*

# Seminar 7 — From Next Token to Assistant: What Each Training Stage Changes

Within the last two weeks we saw how the model evolved inside. New bits and parts of the architecture, but still something felt off. It was still nowhere near the coolest models, or even the smallest Qwen. So, what's the problem?

Every model you actually talk to went through the same sequence: pretraining on trillions of tokens, and then a few rounds of much smaller training — supervised fine-tuning, preference tuning, reinforcement learning — that turn a text continuer into something that answers, refuses, and thinks before it answers.

We will look at every round on one model: **OLMo 2 1B**, which Ai2 released after *each* stage, together with the data of each stage. Four checkpoints, one architecture. Like Pythia last week, that is an experiment somebody already paid for; even if we train almost nothing ourselves, we'll still get the best out of someone's results.

The second half is the question you will meet at work much sooner than the first one: if a few thousand examples can turn a base model into an assistant, why is *"let's just fine-tune it"* so often the wrong answer?

So, two questions for today:

* **What does each stage of training change — and where, if the architecture never changes?**
* **When is fine-tuning a model the wrong move?**

### The claim for today:

$$\mathcal{L}(\theta) \;=\; -\sum_{t\,\in\,\text{trained tokens}} w_t \,\log \pi_\theta\big(y_t \mid y_{<t}\big)$$

Every stage is this one sum — week 2's next-token loss. What changes from stage to stage is only **whose tokens are in the sum**, and **who decides the weight** $w_t$ of each.

![Four stages, one sum: pretraining sums every token with w = 1; SFT only the written answer; DPO pushes the better answer up and the worse down; RLVR sums the model's own samples, and the checker decides up or down](figures/training_stages.png)

Read the picture top to bottom: the model writes more and more of its own training data, and a human — or a program — only says how good it was.

# Some dependencies

In [ ]:
# Colab setup. If needed, uncomment the following lines to install dependencies and clone the repository.
# The notebook reads weeks 1, 4, 5 and 6 from ../, so clone the whole repo.
#!pip install -q "transformers>=5" datasets accelerate peft
#!git clone -q --branch autumn-2026 https://github.com/xufana/dsba_nlp.git
#%cd dsba_nlp/week07_training_alignment

# OLMo's four checkpoints are about 3 GB each. Run this once before the seminar, so section 1 does not wait for the download:
#from huggingface_hub import snapshot_download
#for name in ["allenai/OLMo-2-0425-1B", "allenai/OLMo-2-0425-1B-SFT", "allenai/OLMo-2-0425-1B-DPO", "allenai/OLMo-2-0425-1B-Instruct"]:
#    snapshot_download(name, allow_patterns=["*.safetensors", "*.json", "*.txt"])

In [ ]:
import json
import math
import re
import time
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from datasets import load_dataset
import transformers
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer
from peft import LoraConfig, get_peft_model

warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()
transformers.logging.disable_progress_bar()

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
# half precision for the 1B models: bf16 where the chip has it, fp16 on a T4, fp32 on the CPU
DTYPE = (torch.bfloat16 if device.type == "mps" or (device.type == "cuda" and torch.cuda.is_bf16_supported(including_emulation=False))
         else torch.float16 if device.type == "cuda" else torch.float32)
print("device:", device, DTYPE)

STAGES = {"base": "allenai/OLMo-2-0425-1B",                # pretraining + mid-training
          "SFT":  "allenai/OLMo-2-0425-1B-SFT",            # + supervised fine-tuning
          "DPO":  "allenai/OLMo-2-0425-1B-DPO",            # + preference tuning
          "RLVR": "allenai/OLMo-2-0425-1B-Instruct"}       # + RL with verifiable rewards: the released chat model
SMALL_BASE = "HuggingFaceTB/SmolLM2-135M"                 # last week's body, for our own SFT run
SMALL_CHAT = "HuggingFaceTB/SmolLM2-135M-Instruct"
THINKER = "Qwen/Qwen3-0.6B"                               # one model that can think or not, by a flag
EDU_JUDGE = "HuggingFaceFW/fineweb-edu-classifier"        # the encoder that chose what FineWeb-Edu kept
GPT2 = "gpt2"                                             # week 5's body
PUNCT = "xufana/dsba-week05-punct"                        # the same gpt2 after week 5, fine-tuned to restore punctuation
SFT_DATA = "allenai/tulu-3-sft-olmo-2-mixture-0225"       # what OLMo's SFT stage read
PREF_DATA = "allenai/olmo-2-0425-1b-preference-mix"       # what OLMo's DPO stage read

W1, W4, W5, W6 = "../week01_text_representations", "../week04_pretraining_transfer", "../week05_decoder_only", "../week06_modern_llm"
ART_DIR = "artifacts"
RECOMPUTE = False                                         # True: rebuild every artifact with precompute.py (about three hours on an M-series Mac)

N_DEMO = 200                                              # texts behind every live measurement on AG News
N_LIVE = 2_000                                            # dialogues in our own SFT run

In [ ]:
SEED = 42
IGNORE = -100
CLASS_WORDS = [" world", " sports", " business", " technology"]       # week 5 §3.0: the verbalizer that won
PROMPTS = ["Find the value of n in the equation n + 2 = 6.",
           "Give me three tips for my first conference talk.",
           "How can I get into my neighbour's wifi without the password?"]


def n_params(module):
    return sum(p.numel() for p in module.parameters())


PEAK = 0.0


def gpu_gb():
    """What the GPU holds *right now*, in GB — reserved, cache included, so it is comparable to the device's total.
    Apple's backend has no peak counter, so PEAK is sampled here by whoever calls this."""
    global PEAK
    now = (torch.mps.driver_allocated_memory() / 1e9 if torch.backends.mps.is_available() else
           torch.cuda.memory_reserved() / 1e9 if torch.cuda.is_available() else 0.0)
    PEAK = max(PEAK, now)
    return now


def free():
    """Give the allocator's cache back. `del` the model first — this cannot reach the caller's names."""
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()
    elif torch.cuda.is_available():
        torch.cuda.empty_cache()


def chunks(tag, n, size):
    """`range(0, n, size)`, plus one rewritten line with the rate, the estimate and the GPU — these loops run for minutes."""
    t0 = time.time()
    for done, i in enumerate(range(0, n, size)):
        yield i
        seen = min(i + size, n)
        if done % 10 == 9 or seen == n:
            rate = seen / max(time.time() - t0, 1e-9)
            print(f"\r  {tag} {seen:,}/{n:,}  {rate:,.1f}/s  eta {(n - seen) / rate:4.0f}s  gpu {gpu_gb():4.1f} GB   ",
                  end="", flush=True)
    print(flush=True)

# 0. Where we stopped

## 0.1 The data, and the tables we keep

AG News, the same test set; the results table comes back with its rows from weeks 1, 4, 5 and 6. Today it gets one more row — and not a good one.

In [ ]:
def target_text(text):
    """Week 5's clean-up: the original, minus AG News' line-break and entity artifacts ("\\", "#36;")
    and minus tokens that are punctuation only ("-", "&")."""
    text = text.replace("\\", " ").replace("#36;", "$")
    return " ".join(w for w in text.split() if re.search(r"[A-Za-z0-9]", w))


def strip_text(text):
    """What ASR hands you: lower case, no punctuation. Apostrophes stay — they are pronounced."""
    return re.sub(r"[^a-z0-9' ]", "", text.lower())


ds = load_dataset("fancyzhx/ag_news")
demo_texts = [target_text(t) for t in ds["test"]["text"][:N_DEMO]]
gold = np.array(ds["test"]["label"][:N_DEMO])

results = []


def record(model, test_acc, note=""):
    results.append({"model": model, "test acc": round(test_acc, 4), "note": note})


def show_results():
    return pd.DataFrame(results).set_index("model")


week1 = pd.read_csv(f"{W1}/artifacts/reference_results.csv")
tfidf = week1[week1.representation.str.startswith("tf-idf")].sort_values("accuracy").iloc[-1]
record("TF-IDF + logreg (week 1)", tfidf.accuracy, tfidf.representation)
w4_ft = json.load(open(f"{W4}/artifacts/agnews_transfer.json"))
record("fine-tuned BERT, everything (week 4)", w4_ft["runs"]["everything"]["accuracy"][-1], f"{w4_ft['n_train']:,} texts")
w6_family = json.load(open(f"{W6}/artifacts/family.json"))["models"]
for name in ["gpt2", "SmolLM2-135M", "SmolLM2-1.7B"]:
    record(f"zero-shot scorer, {name} (weeks 5–6)", w6_family[name]["zero_shot"]["accuracy"], "4 passes per text, no training")
show_results()

## 0.2 The stage table

Week 5 kept a cost table, week 6 a bill. Today's table has one row per **stage of one model**: the four OLMo checkpoints. Every section adds a column — and by the end the table is the seminar.

**Bet, for the whole seminar:** you have a good open base model and 1 000 labelled examples of your task. Do you fine-tune it? Write down *yes* or *no* and one reason. We come back to it in 6.3.

In [ ]:
stages = {}


def note(stage, **columns):
    stages.setdefault(stage, {}).update(columns)


def show_stages():
    return pd.DataFrame(stages).T

# 1. Four checkpoints of one model

OLMo 2 1B (Team OLMo 2024; the 1B came out in April 2025) is a fully open model: the weights, the code, and the data of every stage. Its training, stage by stage:

1. **base** — pretraining on about four trillion tokens of web, code, papers and maths, then *mid-training*: a last ~50B tokens of selected high-quality text and maths, with the learning rate taken down to zero (section 2). <!-- CHECK: token counts for the 1B, OLMo 2 paper / model card -->
2. **SFT** — supervised fine-tuning on the Tülu 3 dialogue mixture, adapted to OLMo (`allenai/tulu-3-sft-olmo-2-mixture-0225`).
3. **DPO** — preference tuning on 378k pairs of answers, one marked better (`allenai/olmo-2-0425-1b-preference-mix`).
4. **RLVR** — reinforcement learning with verifiable rewards on maths problems with a known answer (`allenai/RLVR-MATH`). `-Instruct` is the checkpoint after the last round of it. <!-- CHECK: -RLVR1 vs -Instruct -->

Same code, same shapes; four sets of numbers.

**Make a guess** before the cell runs. Three prompts go to each checkpoint: a one-line equation, a request for advice, and a request to break into someone's wifi. What does the **base** model do with them? At which stage does the wifi request first get a refusal?

In [ ]:
def as_prompt(tokenizer, question, chat=True, **template):
    """A chat model gets the conversation through its template. A base model has no template: it gets text to continue."""
    if chat:
        return tokenizer.apply_chat_template([{"role": "user", "content": question}], add_generation_prompt=True, tokenize=False, **template)
    return f"Question: {question}\nAnswer:"


@torch.no_grad()
def generate(model, tokenizer, prompts, max_new_tokens, n=1, temperature=0.0, top_p=1.0, batch_size=16):
    """Finished prompt strings in, `n` answers per prompt out: [{"text", "tokens", "stopped"}, …] per prompt.
    temperature 0 is greedy. `stopped` says whether the model ended the answer itself, before max_new_tokens."""
    tokenizer.padding_side = "left"
    tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token          # gpt2 has none
    stop = model.generation_config.eos_token_id
    stops = sorted({tokenizer.eos_token_id, *(stop if isinstance(stop, list) else [stop] if stop is not None else [])})
    sampling = {"do_sample": True, "temperature": temperature, "top_p": top_p} if temperature > 0 else {"do_sample": False}
    out = []
    for i in chunks("generate", len(prompts), batch_size):
        enc = tokenizer(prompts[i:i + batch_size], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        ids = model.generate(**enc, max_new_tokens=max_new_tokens, num_return_sequences=n, eos_token_id=stops,
                             pad_token_id=tokenizer.pad_token_id, **sampling)
        new = ids[:, enc.input_ids.shape[1]:].tolist()
        for j in range(0, len(new), n):
            group = []
            for seq in new[j:j + n]:
                end = next((k for k, t in enumerate(seq) if t in stops), None)
                body = seq if end is None else seq[:end]
                group.append({"text": tokenizer.decode(body, skip_special_tokens=True), "tokens": len(body), "stopped": end is not None})
            out.append(group)
    return out

In [ ]:
@torch.no_grad()
def bits_per_byte(model, tokenizer, texts, batch_size=16):
    """Next-token loss in bits per UTF-8 byte of text — the unit that survives a change of tokenizer.
    Every text starts from <bos>, so every byte of it is predicted by something."""
    start = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else tokenizer.eos_token_id
    nats = 0.0
    for i in range(0, len(texts), batch_size):
        seqs = [[start] + tokenizer(t, add_special_tokens=False).input_ids for t in texts[i:i + batch_size]]
        width = max(map(len, seqs))
        ids = torch.tensor([s + [start] * (width - len(s)) for s in seqs], device=model.device)
        mask = torch.tensor([[1] * len(s) + [0] * (width - len(s)) for s in seqs], device=model.device)
        logits = model(input_ids=ids, attention_mask=mask).logits[:, :-1].float()
        nats += (F.cross_entropy(logits.transpose(1, 2), ids[:, 1:], reduction="none") * mask[:, 1:]).sum().item()
    return nats / np.log(2) / sum(len(t.encode()) for t in texts)

In [ ]:
answers, configs = {}, {}
for stage, name in STAGES.items():
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, dtype=DTYPE).to(device).eval()
    configs[stage] = model.config.to_dict()
    chat = stage != "base"                                  # a base model has no template: it gets text to continue
    answers[stage] = [a[0]["text"] for a in generate(model, tok, [as_prompt(tok, q, chat) for q in PROMPTS], 160)]
    note(stage, parameters=f"{n_params(model):,}", **{"bits / byte": round(bits_per_byte(model, tok, demo_texts), 4)})
    del model
    free()

differ = sorted(k for k in configs["base"] if any(configs[s].get(k) != configs["base"][k] for s in configs))
print("config keys that differ between the four checkpoints:", differ)
for i, q in enumerate(PROMPTS):
    print(f"\n=== {q}")
    for stage in STAGES:
        print(f"[{stage:>4}] {answers[stage][i][:420]!r}")

<!-- CHECK LIVE: what each stage answered; which keys differ (expect only the name / path and generation settings); the bits per byte; the template printed below. -->

Read the answers stage by stage.

**base** does not answer anything; it *continues*. Given `Question: … Answer:` it writes something that looks like the rest of a page with questions and answers on it — often the right number, then the next question it made up. That is all pretraining asked of it: produce the most likely next text. The knowledge is there (the equation is trivial for a model that read four trillion tokens); what is missing is any idea that *we* are someone, and that the text should end when our question is answered.

**SFT** already answers, in the form of an assistant, and stops. **DPO** and **RLVR** answer in the same form; look at the length, and at the third prompt — where did the refusal come from, if no stage was "taught to refuse" in the table above? (It is in the data: dialogues where the written answer *is* a refusal, and pairs where the refusal was marked better.)

And the first line of the output: the configs are identical. Not one parameter was added, no head was put on (week 4), the architecture of week 6 is untouched — `parameters` in the table is the same number four times. Everything an "assistant" is lives in the numbers inside the same matrices.

The last column is the first price: **bits per byte on AG News**, the language-modelling quality of week 6 §4.1. Every stage after the base is *worse* at predicting news. The post-trained model is no longer optimised to predict text — it is optimised to answer — and the loss we measured in week 6 goes up a little. Hold on to that; it is section 6 in miniature.

### The template

What turns a list of messages into the prefix the model reads is a small program shipped with the tokenizer — the chat template from week 6 §4.3. It is the only "interface" a chat model has:

In [ ]:
print(tok.apply_chat_template([{"role": "user", "content": PROMPTS[0]}], add_generation_prompt=True, tokenize=False))

A few markers and newlines. SFT taught the model what follows `<|assistant|>` and where to put the end-of-text token; the base model has never seen these markers used this way, which is why we gave it `Question: … Answer:` instead — the format that last year's notebook found by trial and error ("Answer: n = "), and that week 5 called *the task lives in the prefix*. SFT is what makes that prefix the same for every task.

## 1.1 How far did the weights move?

**Make a guess:** measure the distance between consecutive checkpoints, $\lVert W_{\text{after}} - W_{\text{before}} \rVert / \lVert W_{\text{before}} \rVert$ over all weights. Which stage moved the model the most — SFT (hundreds of thousands of dialogues), DPO (378k pairs) or RLVR (thousands of problems)? And what share of the 1.5B numbers did each stage leave **exactly** as they were?

In [ ]:
if RECOMPUTE:
    !python precompute.py --only weights

weights = json.load(open(f"{ART_DIR}/weights.json"))
moves = ["base -> SFT", "SFT -> DPO", "DPO -> RLVR"]
display(pd.DataFrame({m: {"relative change": weights["pairs"][m]["relative"], "numbers unchanged": weights["pairs"][m]["same"]} for m in moves}).T)

fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.6))
for m in moves:
    t = pd.DataFrame(weights["pairs"][m]["tensors"])
    t = t[t.layer >= 0].groupby("layer")[["d2", "n2"]].sum()
    ax[0].semilogy(t.index, np.sqrt(t.d2 / t.n2), marker="o", label=m)
    k = pd.DataFrame(weights["pairs"][m]["tensors"]).groupby("kind")[["d2", "n2"]].sum()
    ax[1].bar([f"{i}\n{m.split(' ')[-1]}" for i in k.index], np.sqrt(k.d2 / k.n2), alpha=0.7)
ax[0].set(xlabel="layer", ylabel="‖ΔW‖ / ‖W‖", title="how far each stage moved each layer"); ax[0].legend(fontsize=8)
ax[1].set(yscale="log", title="by kind of matrix"); ax[1].tick_params(axis="x", labelsize=6, rotation=90)
plt.tight_layout()

<!-- REVISE AFTER PRECOMPUTE: the three relative changes, the share of identical numbers (expect RLVR to leave a large share bit-for-bit untouched), which layers and matrices moved most. -->

The order of the three bars is the order of the table in the claim: SFT moves the weights furthest, preference tuning less, RL least — and RL leaves a large share of the numbers *bit for bit* the same. That last observation has a paper of its own: RL fine-tuning updates a small subnetwork, and much of the rest is not touched at all (Mukherjee et al. 2025). The reason is in the data: SFT pushes the model towards somebody else's text, token by token; RL only re-weights answers the model already writes itself, so there is less to change.

Put the two halves of section 1 together: the behaviour moved from "continue a web page" to "answer, stop, sometimes refuse", while the weights moved by a few percent. **Post-training is a small move in weight space and a large move in behaviour.** That is why it is cheap — and, section 6, why it is easy to undo.

Also, the base model knew how to solve `n + 2 = 6` before any of this. Zhou et al. (2023) put it as a hypothesis — LIMA, *less is more for alignment*: almost all of a model's knowledge comes from pretraining, and alignment mostly teaches it *which format and style* to use, which is why 1 000 carefully chosen dialogues got them surprisingly far. Keep the hypothesis in mind; the next sections test it from three sides.

# 2. Pretraining, briefly: what the tokens are

The compute of pretraining was last week's bill: $6ND$, and a law for splitting it. What week 6 did not ask is **which** $D$ tokens. By 2024 the answer to "how do we make the base model better at a fixed $N$ and $D$" was mostly *data*: deduplicate the web, throw away most of it, and put the best text at the end.

The best known filter is FineWeb-Edu (Penedo et al. 2024). Llama-3-70B was asked to rate 450 000 web pages for *educational value* on a scale of 0–5; a small **encoder** with a regression head was trained on those ratings — week 4's recipe exactly — and then run over the whole of FineWeb, fifteen trillion tokens. Everything below 3 was thrown away. About 1.3 trillion tokens survived, less than a tenth, and models trained on them beat models trained on all fifteen.

**Make a guess:** sixteen random pages of the web. How many does the classifier keep?

In [ ]:
web = load_dataset("HuggingFaceFW/fineweb", name="sample-10BT", split="train", streaming=True)
docs = [row["text"] for row, _ in zip(web, range(16))]
judge_tok = AutoTokenizer.from_pretrained(EDU_JUDGE)
judge = AutoModelForSequenceClassification.from_pretrained(EDU_JUDGE).to(device).eval()
with torch.no_grad():
    enc = judge_tok(docs, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    edu = judge(**enc).logits.squeeze(-1).float().cpu().numpy()
print(f"the judge: {judge.config.model_type}, {n_params(judge) / 1e6:.0f}M parameters, one number out\n")
for s, doc in sorted(zip(edu, docs), key=lambda x: -x[0]):
    print(f"{s:4.1f}  {'KEEP' if round(min(max(s, 0), 5)) >= 3 else '    '}  {doc[:150]!r}")
del judge
free()

<!-- CHECK LIVE: how many of the sixteen pass. -->

Read the bottom of the list first: shop pages, forum noise, adverts — text that is perfectly good *language* and that a 1B model has no reason to spend its capacity on. The few at the top read like a textbook. An encoder with one number on top of it decided what a trillion-token decoder reads; the cheapest body in the course choosing the diet of the most expensive one is a good picture of 2026.

**Mid-training** is the same idea moved along the timeline. The last few percent of pretraining tokens — for OLMo 2, a mix called Dolmino — are the best data available: filtered web, maths, papers, encyclopaedias, and *some instruction-like and question–answer text*, with the learning rate decayed to zero. So the boundary between "pretraining" and "fine-tuning" is blurry: a base model of 2025 has already seen plenty of questions followed by answers. That is why it could solve the equation in section 1, and why SFT needs hundreds of thousands of dialogues and not billions.

# 3. Supervised fine-tuning: the same loss, with a mask

SFT is the cross-entropy of week 2 on dialogues, with one change. The model reads the whole conversation, but the loss is summed only over the tokens of the **answer** — every token of the prompt, the template's markers included, gets the label `IGNORE = -100`, the same value week 4 put on the padding and on the second pieces of words in NER. The model is not asked to predict what the user will write; it is asked to predict what the assistant writes, given everything before it.

The data is OLMo's own SFT mixture. Look at one example before anything else.

In [ ]:
def sft_example(tokenizer, messages, max_len=512, mask=True):
    """One conversation -> input_ids and labels. With mask=True the loss is only on the last answer:
    every token of the prompt (template included) gets IGNORE, exactly like the padding of week 4's NER."""
    prompt = tokenizer.apply_chat_template(messages[:-1], add_generation_prompt=True, tokenize=False)
    full = tokenizer.apply_chat_template(messages, tokenize=False)
    a = tokenizer(prompt, add_special_tokens=False).input_ids
    b = tokenizer(full[len(prompt):], add_special_tokens=False).input_ids
    ids = (a + b)[:max_len]
    labels = ([IGNORE] * len(a) if mask else a) + b
    return {"input_ids": ids, "labels": labels[:max_len]}


def batches(examples, batch_size, pad_id, device):
    """Right-padded batches; padding gets IGNORE in the labels, so it never carries loss, and 0 in the attention mask.
    The mask is built from lengths: a pad id can also be a real token of the template (SmolLM2 pads with <|im_end|>)."""
    for i in range(0, len(examples), batch_size):
        chunk = examples[i:i + batch_size]
        width = max(len(e["input_ids"]) for e in chunk)
        pad = [width - len(e["input_ids"]) for e in chunk]
        ids = torch.tensor([e["input_ids"] + [pad_id] * p for e, p in zip(chunk, pad)], device=device)
        mask = torch.tensor([[1] * len(e["input_ids"]) + [0] * p for e, p in zip(chunk, pad)], device=device)
        labels = torch.tensor([e["labels"] + [IGNORE] * p for e, p in zip(chunk, pad)], device=device)
        yield ids, mask, labels


def train_sft(model, examples, pad_id, epochs=1, lr=2e-4, batch_size=16, every=None, on_step=None):
    """The whole of supervised fine-tuning: next-token cross-entropy on the labels, AdamW, a linear decay.
    Only parameters with requires_grad train — with LoRA that is the adapters. Returns the loss per step."""
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    total = epochs * math.ceil(len(examples) / batch_size)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 1 - s / total)
    losses = []
    model.train()
    for _ in range(epochs):
        for ids, attn, labels in batches(examples, batch_size, pad_id, model.device):
            loss = model(input_ids=ids, attention_mask=attn, labels=labels).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step(); sched.step(); opt.zero_grad()
            losses.append(loss.item())
            if every and on_step and len(losses) % every == 0:
                model.eval(); on_step(len(losses)); model.train()
    model.eval()
    return losses

In [ ]:
stream = load_dataset(SFT_DATA, split="train", streaming=True).shuffle(seed=SEED, buffer_size=10_000)
dialogues = []
for row in stream:
    m = row["messages"]
    if len(m) == 2 and m[0]["role"] == "user" and len(m[0]["content"]) + len(m[1]["content"]) < 2000:   # one question, one answer, short
        dialogues.append(m)
    if len(dialogues) == N_LIVE + 50:
        break
train_dialogues, held_dialogues = dialogues[:N_LIVE], dialogues[N_LIVE:]

small_tok = AutoTokenizer.from_pretrained(SMALL_CHAT)      # the -Instruct tokenizer: the same vocabulary as the base, plus a template
example = sft_example(small_tok, train_dialogues[0])
shown = "".join(("\033[1;32m" if l != IGNORE else "") + small_tok.decode([t]) + "\033[0m"
                for t, l in zip(example["input_ids"], example["labels"]))
print(shown)
print(f"\n{len(example['input_ids'])} tokens, of them carry loss: {sum(l != IGNORE for l in example['labels'])} (green)")

Green is what the model is trained to write: the answer, and the `<|im_end|>` after it — the only reason a fine-tuned model ever *stops*. Everything else is context.

**Let's calculate:**

1. A dialogue: the prompt with its template is 300 tokens, the answer 50 (the end marker included). How many training signals does it give with the loss on every token, the way pretraining reads text? How many with the mask? What share of the forward pass's work is "wasted" on tokens that carry no loss?
2. Forget the mask: train on every token. What does the model learn *besides* answering? What will it do when you give it the start of a conversation?

<details>
<summary>Answer</summary>

1. 349 signals (every position predicts the next token) against 50 — one in seven. The other six sevenths of the pass are still computed: the answer cannot be read without them. That is the price of conditioning.
2. To write the user's side too. It learns the distribution of *questions* and of the template's markers as much as of answers; asked something, it may answer and then carry on with a new `user` turn of its own. With 300 prompt tokens to 50 answer tokens, most of what it learns is how users write.

</details>

Now let's do it, small. Our body is last week's SmolLM2-135M base, the data is 2 000 dialogues from OLMo's mixture, and the method is **LoRA** (Hu et al. 2021): every weight matrix $W$ stays frozen, and next to it trains a product $BA$ of two thin matrices, rank $r = 16$, so that the layer computes $Wx + BAx$. $B$ starts at zero, so at step 0 the model is exactly the base. Week 6's bill, applied to training: the optimiser keeps two numbers for every parameter it trains, so training 1% of them is what makes a fine-tune fit in a notebook.

**Make a guess:** before training, how many of 50 held-out questions does the base model answer *and then stop*? After 125 steps of 16 dialogues?

In [ ]:
small = AutoModelForCausalLM.from_pretrained(SMALL_BASE, dtype=torch.float32).to(device)
held_q = [as_prompt(small_tok, d[0]["content"]) for d in held_dialogues]
before = [a[0] for a in generate(small, small_tok, held_q, 128)]
bpb_before = bits_per_byte(small, small_tok, demo_texts)

small = get_peft_model(small, LoraConfig(r=16, lora_alpha=32, target_modules="all-linear", task_type="CAUSAL_LM"))
trainable = sum(p.numel() for p in small.parameters() if p.requires_grad)
print(f"trainable: {trainable:,} of {n_params(small):,} ({trainable / n_params(small):.1%})")
t = time.time()
losses = train_sft(small, [sft_example(small_tok, d) for d in train_dialogues], small_tok.pad_token_id)
print(f"{len(losses)} steps in {time.time() - t:.0f}s")
after = [a[0] for a in generate(small, small_tok, held_q, 128)]
bpb_after = bits_per_byte(small, small_tok, demo_texts)

display(pd.DataFrame({"base": {"stops by itself": np.mean([a["stopped"] for a in before]), "answer, tokens": np.mean([a["tokens"] for a in before]), "bits / byte on AG News": bpb_before},
                      "base + our SFT": {"stops by itself": np.mean([a["stopped"] for a in after]), "answer, tokens": np.mean([a["tokens"] for a in after]), "bits / byte on AG News": bpb_after}}).round(3))
plt.figure(figsize=(5, 2.6)); plt.plot(losses); plt.xlabel("step"); plt.ylabel("loss on the answers"); plt.tight_layout()
for i in range(2):
    print(f"\n### {held_dialogues[i][0]['content'][:200]!r}\nbase:  {before[i]['text'][:300]!r}\nSFT:   {after[i]['text'][:300]!r}")

<!-- CHECK LIVE: the two "stops by itself" shares, the loss curve, the two bits-per-byte numbers. -->

Two minutes, one percent of the parameters, two thousand examples: the base model, which wrote until we cut it off, now answers in the assistant's format and ends the turn. That is the format part of LIMA's hypothesis, measured. Whether the answers are any *good* is a different matter — read the two examples; a 135M model knows what 135M parameters can hold, and SFT does not add to that.

And the last row of the table: bits per byte on news went up. We trained on 2 000 dialogues and paid for it on a task nobody asked about. Keep the number for section 6.

The precomputed run does the same at ten times the data, with and without the mask, next to the official SmolLM2-135M-Instruct:

In [ ]:
if RECOMPUTE:
    !python precompute.py --only sft

sft = json.load(open(f"{ART_DIR}/sft.json"))
rows = {"base": sft["before"], **{f"SFT, loss on {k}": v for k, v in sft["runs"].items()}, "official SmolLM2-135M-Instruct": sft["official instruct"]}
display(pd.DataFrame({k: {"stops by itself": v["stopped"], "answer, tokens": v["tokens"], "bits / byte": v["bits_per_byte"],
                          "tokens with loss": v.get("share_with_loss", "")} for k, v in rows.items()}).T)
print(f"{sft['n_train']:,} dialogues, LoRA r = {sft['lora_r']}, {sft['device']}\n")
for k in sft["runs"]:
    print(f"[{k}] {sft['runs'][k]['demo'][0][:400]!r}\n")

<!-- REVISE AFTER PRECOMPUTE: the mask's effect (expect the unmasked run to continue with invented user turns), the gap to the official model. -->

The unmasked run spent most of its updates learning to write *questions* — the share of tokens with loss says how much — and it shows in what it writes. The official model is better than either of ours: its SFT data (SmolTalk, Allal et al. 2025) and its training were a full stage, not two thousand examples; and some of its quality was put in before SFT, by mid-training.

Where does SFT data come from in 2026? Human-written answers, as in InstructGPT (Ouyang et al. 2022), are rare and expensive. Since Alpaca (2023) most of it is **written by a stronger model** — the Tülu 3 mixture in our cell is largely that, plus human-written prompts and filtered public datasets. SFT is mostly *distillation of a teacher's style*, and a model fine-tuned on a teacher's answers learns the teacher's *format* much faster than its knowledge (Gudibande et al. 2023): the imitation sounds right before it is right.

# 4. Preferences: from RLHF to DPO

SFT has a limit that is built into its loss: it only ever says *"write this"*. It never says *"not that"*. And for most of what we want from an assistant — helpful but not wordy, confident but not invented, polite but not grovelling — it is hard to write the best answer, and easy to say which of two answers is better. So the next stage learns from **comparisons**.

## 4.1 RLHF, and Bradley–Terry

InstructGPT (Ouyang et al. 2022), following Christiano et al. (2017), did it in two steps.

**First, a reward model.** People see a prompt and two answers and pick one. To learn from picks we need a model of how a pick happens; the one used is from 1952 (Bradley & Terry, ranking chess players and tournaments): every answer has a hidden score $r$, and

$$P(\text{A is preferred to B}) \;=\; \sigma\big(r(x, A) - r(x, B)\big).$$

Only the *difference* of scores matters. A network — the SFT model with its vocabulary head swapped for one number, week 4's regression head — is trained on the picks with the loss $-\log\sigma(r_{chosen} - r_{rejected})$: logistic regression on the difference, the classifier of week 1, applied to a pair. (Chatbot Arena's leaderboard is the same model fitted to votes over pairs of *systems*: one formula ranks answers for training and models for evaluation.)

**Then, reinforcement learning.** The SFT model generates, the reward model scores, and PPO (Schulman et al. 2017) pushes up the answers with high reward — with a penalty for drifting away from the SFT model, $\beta \cdot \mathrm{KL}(\pi \,\|\, \pi_{\text{SFT}})$, so that it does not find the reward model's blind spots. The objective is

$$\max_\pi \; \mathbb{E}_{y \sim \pi}\big[r(x, y)\big] \;-\; \beta\, \mathrm{KL}\big(\pi(\cdot|x) \,\|\, \pi_{\text{ref}}(\cdot|x)\big).$$

Count what PPO keeps in memory, in week 6's terms: the policy being trained, the frozen reference, the reward model, and a value network ("critic") to estimate how good a half-written answer is — **four models of the size of the policy**, two of them with optimiser states, and generation inside the training loop. It worked; it was famously hard to make work.

**Let's calculate (a):** the reward model gives the chosen answer $r = 2.0$ and the rejected one $r = 0.5$. The probability it assigns to the human's pick? The loss on this pair?

## 4.2 DPO: the reward model was inside the policy

Rafailov et al. (2023) noticed that the objective above has an exact solution. For a fixed reward, the best policy is the reference re-weighted by the exponentiated reward:

$$\pi^*(y|x) = \frac{1}{Z(x)}\, \pi_{\text{ref}}(y|x)\, e^{\,r(x,y)/\beta} \quad\Longleftrightarrow\quad r(x,y) = \beta \log\frac{\pi^*(y|x)}{\pi_{\text{ref}}(y|x)} + \beta \log Z(x).$$

$Z(x)$ is a sum over *every possible answer* — nobody can compute it. But put this $r$ into Bradley–Terry: both answers of a pair answer the same prompt, so $\beta \log Z(x)$ appears in both scores and **cancels in the difference**. What is left contains only log-probabilities that we can compute:

$$\mathcal{L}_{\text{DPO}} = -\log \sigma\Big(\beta \Big[\underbrace{\log\tfrac{\pi(y_c|x)}{\pi_{\text{ref}}(y_c|x)}}_{\text{how much more likely the chosen answer became}} - \underbrace{\log\tfrac{\pi(y_r|x)}{\pi_{\text{ref}}(y_r|x)}}_{\text{the same for the rejected one}}\Big]\Big).$$

No reward model, no sampling, no critic; two models in memory (the one trained and the frozen SFT), and a loss you can write in one line. The paper's subtitle says what happened: *your language model is secretly a reward model* — $\beta \log \pi/\pi_{\text{ref}}$ **is** the reward, and Bradley–Terry is what lets $Z$ disappear. This is why the 2023–2025 open models, OLMo 2 among them, used DPO and not PPO for this stage.

**Let's calculate (b), (c):** $\beta = 0.1$.

* (b) The reference (SFT) gives the chosen answer $\log \pi_{\text{ref}} = -40$ and the rejected one $-42$. After training, the policy gives them $-35$ and $-52$. The implicit rewards, the margin, the loss? Compare with (a).
* (c) Another policy gives them $-45$ and $-62$. The loss? Did the chosen answer become *more* likely?

In [ ]:
beta = 0.1
dpo_margin = lambda pc, pr, rc, rr: beta * ((pc - rc) - (pr - rr))
bt_loss = lambda margin: -F.logsigmoid(torch.tensor(margin)).item()
print(f"(a) reward model: P = {torch.sigmoid(torch.tensor(2.0 - 0.5)):.3f}, loss = {bt_loss(2.0 - 0.5):.3f}")
for name, pc, pr in [("(b)", -35, -52), ("(c)", -45, -62)]:
    m = dpo_margin(pc, pr, -40, -42)
    print(f"{name} rewards {beta * (pc + 40):+.1f} / {beta * (pr + 42):+.1f}, margin {m:.2f}, loss {bt_loss(m):.3f}")

(a) $\sigma(1.5) = 0.82$, loss $0.20$. (b) The rewards are $+0.5$ and $-1.0$, the margin $1.5$ — the same number as (a), so the same loss: DPO is the reward model's loss with the reward read off the policy. (c) The same margin, the same loss — and the chosen answer became $e^5 \approx 150$ times *less* likely. The loss only sees the **difference**; it is perfectly happy when both answers go down, as long as the rejected one goes down faster. Whether that happens on a real model is a measurement. Let's make it.

## 4.3 The implicit reward of a real checkpoint

Take pairs from OLMo's own DPO data and compute $\log \pi$ of each answer under the SFT model (DPO's reference) and under the DPO model. No training — four numbers per pair, and the sign of the margin.

**Make a guess:** on what share of the pairs does the DPO model prefer the chosen answer *more* than SFT did? And did $\log \pi(\text{chosen})$ go up or down?

In [ ]:
@torch.no_grad()
def answer_logprob(model, prefixes, answers, batch_size=8):
    """log P(answer | prefix) for token-id lists, summed over the answer's tokens; and the answer's length.
    Like `score`, the unembedding runs only where an answer token is predicted — the vocabulary is 100k wide."""
    body = model.model
    out = np.zeros((len(prefixes), 2))
    order = sorted(range(len(prefixes)), key=lambda k: len(prefixes[k]) + len(answers[k]))
    for i in range(0, len(order), batch_size):
        idx = order[i:i + batch_size]
        seqs = [prefixes[k] + answers[k] for k in idx]
        width = max(map(len, seqs))
        ids = torch.tensor([s + [0] * (width - len(s)) for s in seqs], device=model.device)
        mask = torch.tensor([[1] * len(s) + [0] * (width - len(s)) for s in seqs], device=model.device)
        h = body(input_ids=ids, attention_mask=mask).last_hidden_state
        for row, k in enumerate(idx):
            a, b = len(prefixes[k]), len(answers[k])
            logits = model.lm_head(h[row, a - 1:a + b - 1]).float()
            out[k] = logits.log_softmax(-1).gather(1, torch.tensor(answers[k], device=model.device)[:, None]).sum().item(), b
    return out

In [ ]:
stream = load_dataset(PREF_DATA, split="train", streaming=True).shuffle(seed=SEED, buffer_size=5_000)
tok = AutoTokenizer.from_pretrained(STAGES["SFT"])           # one tokenizer for all four stages
prefixes, chosen, rejected = [], [], []
for row in stream:
    p = tok(tok.apply_chat_template(row["chosen"][:-1], add_generation_prompt=True, tokenize=False), add_special_tokens=False).input_ids
    c = tok(row["chosen"][-1]["content"] + tok.eos_token, add_special_tokens=False).input_ids
    r = tok(row["rejected"][-1]["content"] + tok.eos_token, add_special_tokens=False).input_ids
    if len(p) + max(len(c), len(r)) <= 768:
        prefixes.append(p); chosen.append(c); rejected.append(r)
    if len(prefixes) == 64:
        break

lp = {}
for stage in ["SFT", "DPO"]:
    model = AutoModelForCausalLM.from_pretrained(STAGES[stage], dtype=DTYPE).to(device).eval()
    lp[stage] = {"chosen": answer_logprob(model, prefixes, chosen)[:, 0], "rejected": answer_logprob(model, prefixes, rejected)[:, 0]}
    del model
    free()
d_c, d_r = lp["DPO"]["chosen"] - lp["SFT"]["chosen"], lp["DPO"]["rejected"] - lp["SFT"]["rejected"]
print(f"{len(prefixes)} pairs: DPO prefers the chosen answer more than SFT did on {np.mean(d_c > d_r):.0%};  "
      f"log π(chosen) moved by {d_c.mean():+.1f} on average, log π(rejected) by {d_r.mean():+.1f}")

In [ ]:
if RECOMPUTE:
    !python precompute.py --only rewards

rw = json.load(open(f"{ART_DIR}/rewards.json"))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for stage, color in [("DPO", "C0"), ("RLVR", "C2")]:
    dc = np.array(rw[stage]["chosen"]) - np.array(rw["SFT"]["chosen"])
    dr = np.array(rw[stage]["rejected"]) - np.array(rw["SFT"]["rejected"])
    ax[0].hist(dc - dr, bins=60, alpha=0.6, color=color, label=f"{stage}: margin > 0 on {np.mean(dc > dr):.0%}")
    ax[1].scatter(dc, dr, s=4, alpha=0.4, color=color, label=stage)
ax[0].axvline(0, color="k", lw=0.8); ax[0].set(xlabel="(Δ log π chosen) − (Δ log π rejected), against SFT", title=f"{rw['n']:,} pairs of the DPO data"); ax[0].legend(fontsize=8)
lim = np.percentile(np.abs(np.r_[dc, dr]), 99)
ax[1].plot([-lim, lim], [-lim, lim], "k", lw=0.6); ax[1].axhline(0, color="gray", lw=0.5); ax[1].axvline(0, color="gray", lw=0.5)
ax[1].set(xlabel="Δ log π(chosen)", ylabel="Δ log π(rejected)", xlim=(-lim, lim), ylim=(-lim, lim), title="below the diagonal: the pair moved the right way"); ax[1].legend(fontsize=8)
plt.tight_layout()

<!-- REVISE AFTER PRECOMPUTE / CHECK LIVE: the share with a positive margin; the sign of the mean change of log π(chosen) — the literature expects it to be negative (both go down); where RLVR sits relative to DPO. -->

Two readings. First, the share: on its own training pairs DPO prefers the chosen answer more than its reference did on most of them — but not all; nobody trains to zero loss on 378k noisy pairs. These are training pairs, so this is training accuracy, not a test.

Second, the right panel, which is (c) on real numbers. Most of the points are below the diagonal (the pair moved the right way), and a large part of them sit in the lower-left quadrant: *both* answers became less likely, the rejected one more so. This is a known property of DPO, not a bug of OLMo: probability has to go somewhere, and the loss does not say where — usually into answers that are in neither half of any pair, including the model's own style. Several of the variants that followed (IPO, SimPO, APO in SmolLM3; Tülu 3 uses a length-normalised DPO) change exactly the function of the margin, or add a term that keeps $\log\pi(\text{chosen})$ from falling. <!-- CHECK: OLMo 2 1B's DPO variant and β -->

Then **length**. Preference data has a well-documented bias: judges, human or model, tend to prefer the longer of two answers, and preference-tuned models learn to write more (Singhal et al. 2023). The stage table has the column:

In [ ]:
if RECOMPUTE:
    !python precompute.py --only stages

st = json.load(open(f"{ART_DIR}/stages.json"))
for stage, row in st["stages"].items():
    note(stage, **{"open prompts: tokens": row["open"]["tokens"], "stops by itself": row["open"]["stopped"],
                   "GSM8K, greedy": row["gsm8k"]["accuracy"], "GSM8K: tokens": row["gsm8k"]["tokens"]})
print(f"{st['n_open']} open prompts and {st['n_gsm']} GSM8K problems, greedy, up to {st['max_new_tokens']} tokens, {st['device']}")
show_stages()

<!-- REVISE AFTER PRECOMPUTE: length by stage; GSM8K by stage. -->

Read the columns one at a time. `stops by itself` separates base from the rest — section 3. `open prompts: tokens` is the length question. `GSM8K, greedy` is the next section's column: which of the stages made the model better at maths, and by how much?

# 5. RL with a verifier, and reasoning

For one kind of task nobody has to compare anything: when the answer can be **checked**. A maths problem with a known number, code with tests, an output that must parse as JSON. Then the reward model of 4.1 is replaced by a program, and the reward cannot be fooled by style or length — only by the check itself. This is *reinforcement learning with verifiable rewards*, RLVR (Lambert et al. 2024 named it for Tülu 3), and it is what OLMo's last stage did on maths problems. It is also how DeepSeek-R1 (2025) learned to reason.

## 5.1 GRPO: the group is the baseline

The algorithm that won is GRPO (Shao et al. 2024, *DeepSeekMath*). For one prompt the model writes a **group** of $G$ answers; each gets its reward $r_i$ (1 if the checker says right, 0 if not); and each answer's **advantage** is how much better it did than its own group:

$$A_i = \frac{r_i - \operatorname{mean}(r_1, \dots, r_G)}{\operatorname{std}(r_1, \dots, r_G)}.$$

Then every token of answer $i$ gets pushed up or down by $A_i$ — the claim's $w_t$ — with the same KL leash to a reference as in PPO. Compare with 4.1: no reward model (a checker), no critic (the group average is the baseline the critic used to estimate). Two of PPO's four models are gone.

**Let's calculate:** $G = 8$, rewards $[1, 0, 0, 1, 0, 0, 0, 1]$. The advantages? What do they sum to? What if all eight are right — or all eight wrong?

In [ ]:
def gold_number(answer):
    """GSM8K writes the reference solution and then `#### 42`."""
    return float(answer.split("####")[-1].strip().replace(",", ""))


def final_number(text):
    """The number a solution ends on: \\boxed{…}, '#### …', 'the answer is …', or else the last number in the text.
    A base model keeps writing the next 'Question:' after its answer, so that part is cut first."""
    text = text.split("\nQuestion:")[0].replace(",", "")
    for pattern in [r"\\boxed\{([^}]*)\}", r"####\s*([^\n]*)", r"answer is[:\s]*([^\n]*)"]:
        found = re.findall(pattern, text)
        if found:
            text = found[-1]
            break
    numbers = re.findall(r"-?\d+(?:\.\d+)?", text)
    return float(numbers[-1]) if numbers else None


def reward(text, gold):
    """The whole 'reward model' of RL with a verifier: 1 if the final number is right, 0 otherwise."""
    pred = final_number(text)
    return float(pred is not None and abs(pred - gold) < 1e-6)


def pass_at_k(n, c, k):
    """Probability that at least one of k answers is right, estimated without bias from n samples with c right
    (Chen et al. 2021): 1 − C(n−c, k) / C(n, k)."""
    if n - c < k:
        return 1.0
    return 1.0 - float(np.prod(1.0 - k / np.arange(n - c + 1, n + 1)))

In [ ]:
def advantages(rewards, eps=1e-6):
    r = np.asarray(rewards, dtype=float)
    return (r - r.mean()) / (r.std() + eps)


for group in [[1, 0, 0, 1, 0, 0, 0, 1], [1] * 8, [0] * 8]:
    a = advantages(group)
    print(group, "->", a.round(2), " sum", round(a.sum(), 6))

$+1.29$ for each right answer, $-0.77$ for each wrong one; they sum to zero — the group as a whole is not pushed anywhere, only re-weighted inside. With eight right or eight wrong every advantage is **zero**: the prompt teaches nothing. Too easy and too hard problems are wasted compute, and DAPO (Yu et al. 2025) simply filters them out of the batch. Dr. GRPO (Liu et al. 2025) went after the other part of the formula: dividing by the std gives the most weight to the problems where the group almost agrees, and averaging the loss over each answer's own length rewards long wrong answers less than short ones — both biases they removed.

## 5.2 One step of GRPO, without the step

RL starts from the DPO checkpoint. Eight GSM8K problems, eight samples each at temperature 0.8, our 15-line checker. That is the batch GRPO would compute — everything except `loss.backward()`.

**Make a guess:** how many of the eight groups teach nothing?

In [ ]:
gsm = load_dataset("openai/gsm8k", "main", split="test")
problems = gsm.select(range(8))
golds = [gold_number(a) for a in problems["answer"]]
tok = AutoTokenizer.from_pretrained(STAGES["DPO"])
model = AutoModelForCausalLM.from_pretrained(STAGES["DPO"], dtype=DTYPE).to(device).eval()
torch.manual_seed(SEED)
groups = generate(model, tok, [as_prompt(tok, q) for q in problems["question"]], 384, n=8, temperature=0.8, batch_size=2)
R = np.array([[reward(a["text"], g) for a in group] for group, g in zip(groups, golds)])
A = np.array([advantages(r) for r in R])
display(pd.DataFrame(R.astype(int), index=[f"problem {i}" for i in range(8)], columns=[f"answer {j}" for j in range(8)])
        .assign(**{"right": R.sum(1).astype(int), "teaches": np.where(np.abs(A).sum(1) > 0, "yes", "nothing")}))
i = int(np.argmax(np.abs(A).sum(1)))                       # a group that does teach something
print(f"problem {i}: {problems['question'][i][:200]!r}  (answer {golds[i]:g})")
for a, adv in zip(groups[i], A[i]):
    print(f"  A = {adv:+.2f}  {a['tokens']:3d} tokens  …{a['text'][-160:]!r}")
del model
free()

<!-- CHECK LIVE: how many groups are all right / all wrong; one group's answers with their advantages. -->

Look at the group that teaches something: the right answers get $A > 0$, and *every token of them* — the reasoning, not only the final number — is pushed up. That is the whole mechanism of "learning to reason by RL": nobody wrote a solution; the model's own solutions that happened to end on the right number become more likely, together with whatever led there.

The easy groups (eight out of eight) and the hopeless ones (zero) contribute nothing. Which means RL can only sharpen what the model already does *sometimes*. The next measurement asks how far that goes.

## 5.3 Sharper, or smarter? pass@k

pass@k is the probability that *at least one* of $k$ answers is right (Chen et al. 2021); `pass_at_k` above estimates it from $n$ samples without bias. pass@1 is how good the model is when you take one answer. pass@k at large $k$ is what the model *can* do, given enough tries.

**Make a guess:** plot pass@k against $k$ for the base model, SFT and RLVR. At $k = 1$ RLVR is on top. Do the curves cross?

In [ ]:
if RECOMPUTE:
    !python precompute.py --only passk

pk = json.load(open(f"{ART_DIR}/passk.json"))
ks = [k for k in [1, 2, 4, 8, 16, 32, 64, 128] if k <= pk["n"]]
plt.figure(figsize=(6, 3.8))
for stage, r in pk["stages"].items():
    curve = [np.mean([pass_at_k(pk["n"], c, k) for c in r["correct"]]) for k in ks]
    plt.semilogx(ks, curve, marker="o", label=stage)
    note(stage, **{"pass@1": round(curve[0], 3), f"pass@{ks[-1]}": round(curve[-1], 3),
                   f"problems that teach nothing ({pk['n']} samples)": round(np.mean([c in (0, pk["n"]) for c in r["correct"]]), 3)})
plt.xticks(ks, ks); plt.xlabel("k: answers per problem"); plt.ylabel("pass@k")
plt.title(f"GSM8K, {pk['n_problems']} problems, {pk['n']} samples at T = {pk['temperature']}"); plt.legend(); plt.tight_layout()
show_stages()

<!-- REVISE AFTER PRECOMPUTE: where the curves cross, if they do; the share of problems that teach nothing for each stage. -->

The left end of the plot is what everyone reports: RLVR is the best model at one try. The right end is the finding of Yue et al. (2025), measured on many models and tasks: as $k$ grows, the base model catches up, and on their benchmarks it eventually *overtakes* the RL-trained one. RL with a verifier mostly moves probability onto correct solutions the model could already sample — it makes the model **more reliable, not more capable**. The problems the base model never solves in many tries, RL also does not solve. Where the crossing is on our 1B, and whether there is one at 32 samples, is in the plot; what is certain is that the gap at large $k$ is far smaller than at $k = 1$.

The last column of the stage table is 5.1's warning, measured: as a model gets better, more problems are solved by *all* of its samples, and those problems stop teaching. RL needs a supply of problems at the edge of what the model can do — which is why the RL datasets of 2025 are curated by difficulty.

## 5.4 Thinking is written tokens

The other thing RL did in 2025 is to make models **think out loud**: given a reward only for the final answer and room to write, DeepSeek-R1-Zero learned by itself to write long chains of reasoning, to check, to backtrack — and its accuracy grew with the length of what it wrote (DeepSeek-AI 2025). Most models of 2026 do this, and many — Qwen3, SmolLM3 — can do it *or not*, by one switch in the chat template.

That is test-time compute: a third dial, next to week 6's $N$ and $D$. You pay for it in week 5's cost-table column, `tokens written`.

**Make a guess:** Qwen3-0.6B on ten GSM8K problems, thinking on and off. How much more accurate with thinking — and how many times more tokens does it write?

In [ ]:
think_tok = AutoTokenizer.from_pretrained(THINKER)
thinker = AutoModelForCausalLM.from_pretrained(THINKER, dtype=DTYPE).to(device).eval()
print(repr(as_prompt(think_tok, "2 + 2?", enable_thinking=False)))              # the switch is a pair of empty <think> tags
live = gsm.select(range(10))
live_gold = [gold_number(a) for a in live["answer"]]
rows = {}
for mode, on, temp, top_p, budget in [("thinking", True, 0.6, 0.95, 1024), ("no thinking", False, 0.7, 0.8, 384)]:
    torch.manual_seed(SEED)
    out = [a[0] for a in generate(thinker, think_tok, [as_prompt(think_tok, q, enable_thinking=on) for q in live["question"]], budget, temperature=temp, top_p=top_p)]
    rows[mode] = {"accuracy": np.mean([reward(a["text"].split("</think>")[-1], g) for a, g in zip(out, live_gold)]),
                  "tokens written": np.mean([a["tokens"] for a in out]), f"ran out of budget": np.mean([not a["stopped"] for a in out])}
del thinker
free()
pd.DataFrame(rows).T.round(2)

In [ ]:
if RECOMPUTE:
    !python precompute.py --only think

th = json.load(open(f"{ART_DIR}/think.json"))
display(pd.DataFrame({m: {"accuracy": r["accuracy"], "tokens, mean": np.mean(r["tokens"]), "tokens, median": np.median(r["tokens"]),
                          "stopped": r["stopped"], "budget": r["max_new_tokens"]} for m, r in th["modes"].items()}).T.round(3))
plt.figure(figsize=(6, 3))
for m, r in th["modes"].items():
    plt.hist(r["tokens"], bins=40, alpha=0.6, label=m)
plt.xlabel("tokens written per answer"); plt.title(f"{th['model']}, {th['n']} GSM8K problems"); plt.legend(); plt.tight_layout()

<!-- REVISE AFTER PRECOMPUTE: accuracy with and without thinking, the ratio of tokens. -->

Same weights, same question — only the template differs, and the model writes several times more tokens before it answers. Week 5's arithmetic applies to every one of them: one forward pass per token, the cache growing (week 6 §3.1). Thinking is the most expensive way to buy accuracy per request, and the cheapest per *parameter*: the 0.6B model with thinking is the price of a larger model without it, paid at inference time instead of training time.

How small models actually get this ability is worth knowing: mostly not by RL of their own. DeepSeek-R1's paper found that for small models, fine-tuning on reasoning traces written by the big model (SFT again — distillation) beat running RL on the small model directly. RL creates the behaviour at scale; SFT copies it down.

# 6. Why fine-tuning is not always good

Everything above made the model better at what each stage was measured on. Section 1 already showed the other side once — post-training made OLMo worse at predicting news. This section is about that other side, on our own model and in the literature, and ends with a decision you will actually have to make.

## 6.1 Our own model forgot

In week 5 we fine-tuned `gpt2` on one task: restore the punctuation and capitalisation of lower-cased text. It learned it — word accuracy 0.90, and it changed fewer than 1% of the words it should not have. The weights are on the hub.

**Make a guess:** the same `gpt2` and our fine-tuned copy, on the two AG News measurements of week 6 — bits per byte, and week 5's zero-shot scorer (`This news is about` + four words). What happened to each?

In [ ]:
@torch.no_grad()
def score(model, tokenizer, prefixes, continuations, batch_size=64):
    """Week 5's scorer, unchanged: log P(continuation | prefix) for every prefix × continuation, as an array
    [n_prefixes, n_continuations, 2] — the sum over the continuation's tokens and the mean per token.
    One forward pass per pair; the unembedding runs only at the positions the continuation is read from."""
    model.eval()
    body = model.transformer if hasattr(model, "transformer") else model.model
    pre = [tokenizer(p).input_ids for p in prefixes]
    con = [tokenizer(c).input_ids for c in continuations]
    pairs = [(a, b) for a in pre for b in con]
    order = sorted(range(len(pairs)), key=lambda k: len(pairs[k][0]))          # like lengths together: less padding per batch
    out = np.zeros((len(pairs), 2))
    for i in range(0, len(pairs), batch_size):
        chunk = [pairs[k] for k in order[i:i + batch_size]]
        seqs = [a + b for a, b in chunk]
        width = max(map(len, seqs))
        ids = torch.tensor([s + [tokenizer.eos_token_id] * (width - len(s)) for s in seqs], device=model.device)
        mask = torch.tensor([[1] * len(s) + [0] * (width - len(s)) for s in seqs], device=model.device)
        h = body(input_ids=ids, attention_mask=mask).last_hidden_state                             # [B, width, d]
        rows = torch.tensor([k for k, (a, b) in enumerate(chunk) for _ in b], device=model.device)
        cols = torch.tensor([t for a, b in chunk for t in range(len(a) - 1, len(a) + len(b) - 1)], device=model.device)
        want = torch.tensor([t for a, b in chunk for t in b], device=model.device)                  # logits at t predict token t+1
        lp = model.lm_head(h[rows, cols]).float().log_softmax(-1).gather(1, want[:, None]).squeeze(1).cpu().numpy()
        j = 0
        for k, (a, b) in zip(order[i:i + batch_size], chunk):
            out[k] = lp[j:j + len(b)].sum(), lp[j:j + len(b)].mean()
            j += len(b)
    return out.reshape(len(prefixes), len(continuations), 2)

In [ ]:
gpt2_tok = AutoTokenizer.from_pretrained(GPT2)
rows = {}
for name, path in [("gpt2", GPT2), ("gpt2 + week 5's punctuation fine-tune", PUNCT)]:
    model = AutoModelForCausalLM.from_pretrained(path, dtype=torch.float32).to(device).eval()
    pred = score(model, gpt2_tok, [t + " This news is about" for t in demo_texts], CLASS_WORDS)[:, :, 0].argmax(1)
    rows[name] = {"bits / byte": bits_per_byte(model, gpt2_tok, demo_texts), "zero-shot accuracy": (pred == gold).mean(),
                  "predicted classes": dict(Counter(pred.tolist()))}
    if path == PUNCT:
        print("its own task, still:", generate(model, gpt2_tok, [strip_text(demo_texts[0]) + " =>"], 60)[0][0]["text"][:200], "\n")
    del model
    free()
record("zero-shot scorer, gpt2 after week 5's fine-tune", rows["gpt2 + week 5's punctuation fine-tune"]["zero-shot accuracy"], f"{N_DEMO} test texts, live")
pd.DataFrame(rows).T

<!-- CHECK LIVE: bits per byte and zero-shot accuracy before and after; which class the fine-tuned model now predicts. -->

This is **catastrophic forgetting** (McCloskey & Cohen 1989, older than everything else in this course): a network trained on a new task overwrites what made it good at the old one. Nobody asked our model to stay a language model of news; the loss of week 5 only counted punctuated copies of the input. It got exactly what it was trained on, and paid for it on everything else.

Is it the method, or the amount? The precomputed run trains the same task three ways — every weight, LoRA with rank 64, LoRA with rank 8 — and measures both sides along the way:

In [ ]:
if RECOMPUTE:
    !python precompute.py --only forgetting

fg = json.load(open(f"{ART_DIR}/forgetting.json"))
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
for run, r in fg["runs"].items():
    c = pd.DataFrame(r["curve"])
    for a, col in zip(ax, ["task loss", "bits_per_byte", "zero-shot"]):
        a.plot(c.step, c[col], marker=".", label=f"{run} ({r['trainable'] / 1e6:.1f}M trained)")
ax[0].set(title="the new task: held-out loss", xlabel="step"); ax[1].set(title="AG News: bits per byte", xlabel="step")
ax[2].set(title="AG News: zero-shot scorer", xlabel="step"); ax[0].legend(fontsize=8)
plt.tight_layout()

<!-- REVISE AFTER PRECOMPUTE: the three runs on all three panels. Expected (Biderman et al. 2024): full fine-tuning learns the task fastest and forgets most; LoRA learns less and forgets less, more so at low rank. -->

The three panels are one trade-off. Training every weight learns the new task fastest and moves furthest from the old one; LoRA, with a few percent of the parameters, learns slower and forgets less — the title of Biderman et al. (2024) is the result: *LoRA learns less and forgets less*. Neither is free. There is no setting of this dial that learns the new task fully and forgets nothing, because they are the same movement of the same weights.

And it connects to 1.1. RL moved OLMo's weights least of the three stages; Shenfeld et al. (2025) measured the general version — **RL forgets less than SFT** at equal gain on the new task, because sampling its own answers keeps the model close to its own distribution, while SFT pulls it to someone else's.

## 6.2 What else breaks

Four results worth knowing before you fine-tune anything, each a single number away from intuition:

* **New knowledge invites invention.** Fine-tuning a model on facts it did not know from pretraining is learned *slowly* — and as it is learned, the model becomes more prone to hallucinate on everything else (Gekhman et al. 2024). The model learns to state things it has no basis for. SFT teaches *how* to answer; it is a poor way to teach *what* is true.
* **Safety is a thin layer.** A few examples are enough to undo refusal training: GPT-3.5 Turbo fine-tuned through the public API on ten harmful examples, for under $0.20, stopped refusing; and fine-tuning on an ordinary, *benign* instruction dataset also weakened its refusals, without anyone intending it (Qi et al. 2023). 1.1 measured why this is possible: the alignment stages moved the weights by a few percent.
* **Narrow training, broad change.** Fine-tune a chat model on one narrow thing — writing code with security holes, without saying so — and it becomes misaligned far outside code: hostile answers to unrelated questions (Betley et al. 2025, *emergent misalignment*). The model does not learn "a task"; it learns *what kind of assistant writes this*, and generalises the character.
* **The distribution is the product.** Our SFT model in section 3 was worse at news after 2 000 dialogues; OLMo after post-training too. Every fine-tune is a trade, and the side you lose is the one you did not measure.

## 6.3 Should I fine-tune? A design card

Week 5's card, with three new questions in the middle. Answer them in order; the first "yes" to the left of the ladder usually ends the discussion.

```
Task / workload:         what, how many per day, how fast
What is wrong now:       the FORMAT or behaviour (length, style, schema, refusals)  |  the KNOWLEDGE (facts, documents)
Can I check an answer?   with a program (tests, a number, a schema)  |  only by comparing  |  only a human
What must not get worse: the abilities I rely on that are not in my data
Ladder:  prompt  →  few-shot in the prefix  →  retrieval (put the knowledge in the prefix)  →  LoRA SFT  →  full SFT  →  preferences / RL
Body:    is this an LLM task at all?  (week 4: an encoder with a head, one pass)
```

Three cards, two minutes each:

1. **Support tickets → one of 10 queues.** 5 000 labelled tickets, 50 000 tickets a day.
2. **An internal assistant must know the company's documentation**, which changes every week.
3. **Answers must be valid JSON of a fixed schema, and shorter.** The model is right about the content but chatty, and one in twenty outputs does not parse.

<details>
<summary>Answers</summary>

1. Not an LLM at all — the output can be enumerated, there are labels, and the volume is large: week 4's encoder with a classification head, one pass per ticket. If you must use a decoder, a scorer over ten labels (week 5), not generation.
2. Retrieval, not fine-tuning. The knowledge changes weekly (every fine-tune would be stale), fine-tuning is a poor way to add facts (6.2, first bullet), and retrieval lets the answer cite its source. Fine-tune, if ever, for the *format* of answers grounded in retrieved text.
3. The format is the part fine-tuning is good at: a small SFT on examples of the right shape — or preference tuning with "parses and short" as the better side, and the parse check is a *verifiable* reward. And before any of it: constrained decoding, which makes invalid JSON impossible by construction (week 13).

</details>

Now the bet from 0.2: a good base model and 1 000 labelled examples. If what you lack is the format — yes, LoRA SFT, and measure what it forgets. If what you lack is knowledge — no. And whichever it is, a 2026 *instruct* model with your examples in the prefix (week 5) is the baseline that fine-tuning has to beat, and it often does not.

# 7. Where this goes

## 7.1 Both tables

In [ ]:
show_results()

In [ ]:
show_stages()

Read the stage table by columns. `parameters` is the same four times: no stage touched the architecture. `bits / byte` gets worse with every stage: post-training is paid for in language modelling. `stops by itself` and `open prompts: tokens` change at SFT and then drift: format comes first, preferences adjust it. `GSM8K` and `pass@1` grow at the end: the stage with a checker improves what can be checked. And `pass@k` is the column that says how much of that is new ability and how much is reliability.

The two questions from the top:

**What does each stage change?** Whose tokens carry the loss and who weighs them. Pretraining: everybody's text, weighed equally — the knowledge. SFT: a written answer — the format. Preferences: a comparison — the style, the length, the refusals. RL with a verifier: the model's own answers, weighed by a checker — reliability on whatever can be checked, and the habit of thinking before answering. The architecture is the same; the weights move by a few percent.

**When is fine-tuning the wrong move?** When the problem is knowledge rather than format; when the abilities you are not measuring matter; when a prompt, a retrieved document or an encoder would do. It is always a trade: what the new data asks for, against everything it does not mention.

The course so far:

| week | body | training | where the task lives | what it costs | AG News |
|---|---|---|---|---|---|
| 1 | TF-IDF, n-grams | none — counting | the head (logreg) | 1 pass | 0.92 |
| 2 | RNN language model | next token | the decoding rule | $T_{out}$ passes | — |
| 3 | transformer encoder–decoder | conditional next token | the pair | $T_{out}$ passes | — |
| 4 | BERT encoder | MLM (+ NSP), then a head | the head | 1 pass | 0.91 fine-tuned |
| 5 | GPT decoder, 124M | next token | the prefix | candidates or $T_{out}$ passes | 0.61 zero-shot |
| 6 | the same decoder, 135M → 1.7B | next token, 15 000 tokens per parameter | the prefix | $6ND$ once; $2N_{active}$ and a cache per token | 0.66–0.75 zero-shot |
| 7 | the same decoder after SFT, DPO, RL | next token on answers; comparisons; a checker | the prefix, as a conversation — and in the weights' habits | a few % of the weights; tokens of thinking | ↓ after fine-tuning |

## 7.2 Five things that transfer

1. **Every stage is next-token loss.** They differ in whose tokens carry it and who decides their weight — look at those two before anything else.
2. **Check which tokens carry the loss** before you train. A missing mask is a model that learns to write your users' questions.
3. **Post-training is a small move in weight space and a large move in behaviour** — which is why it is cheap, and why it is easy to undo.
4. **RL with a verifier sharpens what the model can already sample.** Measure pass@k, not only pass@1, before you call it a new ability.
5. **Every fine-tune costs something you did not measure.** Pick the abilities you rely on and measure them before and after.

Next week: <!-- TODO: the topic of week 8 -->

## Try it yourself

* **The mask, both ways.** Re-run section 3's SFT without the mask (`sft_example(..., mask=False)`) on the same 2 000 dialogues. Generate answers to the held-out questions with a budget of 256 tokens: how often does the model start a new `user` turn of its own?
* **DPO by hand.** Take `SmolLM2-135M-Instruct` as the reference and the policy, 1 000 pairs of OLMo's preference data, LoRA, and write the DPO loss of 4.2 with `answer_logprob` (with gradients). Log the margin and both log-probabilities every 10 steps. Do both go down?
* **GRPO in thirty lines.** Turn 5.2 into training: `SmolLM2-360M-Instruct`, GSM8K train problems, groups of 8, a reward of 1 for a right final number, the loss $-A_i \log \pi(\text{answer}_i)$ averaged over tokens, no KL. Plot pass@1 on 100 test problems every 20 steps. Then remove the std from the advantage (Dr. GRPO) and compare.
* **A thinking budget.** Qwen3-0.6B, thinking on, `max_new_tokens` from 128 to 4 096. Cut every answer that ran out of budget by appending `</think>` and asking for the answer. Plot accuracy against the budget: where is the knee?
* **Rank and forgetting.** Extend 6.1's experiment to LoRA ranks 1, 2, 4, …, 256. For each, the task loss at the end against the AG News accuracy at the end: is there a rank that is on the frontier of both?
* **Undo the forgetting.** Average the weights of `gpt2` and our punctuation model, $\alpha W_{ft} + (1-\alpha) W_{base}$, for $\alpha$ from 0 to 1. Plot the task's loss and the zero-shot accuracy against $\alpha$. Model merging is a real technique in 2026 — why does a straight line between two models work at all?

## References

**The pipeline.** Ouyang et al. (2022), *Training Language Models to Follow Instructions with Human Feedback* — InstructGPT, SFT then RLHF. Team OLMo (2024), *2 OLMo 2 Furious* — the model of this seminar, and its data. Lambert et al. (2024), *Tülu 3: Pushing Frontiers in Open Language Model Post-Training* — the post-training recipe OLMo 2 uses, and the name RLVR. Zhou et al. (2023), *LIMA: Less Is More for Alignment* — the superficial alignment hypothesis.

**Data.** Penedo et al. (2024), *The FineWeb Datasets: Decanting the Web for the Finest Text Data at Scale* — FineWeb-Edu and its classifier. Allal et al. (2025), *SmolLM2* — SmolTalk. Taori et al. (2023), *Alpaca*. Gudibande et al. (2023), *The False Promise of Imitating Proprietary LLMs*.

**SFT.** Hu et al. (2021), *LoRA: Low-Rank Adaptation of Large Language Models*.

**Preferences.** Bradley & Terry (1952), *Rank Analysis of Incomplete Block Designs*. Christiano et al. (2017), *Deep Reinforcement Learning from Human Preferences*. Schulman et al. (2017), *Proximal Policy Optimization Algorithms*. Rafailov et al. (2023), *Direct Preference Optimization: Your Language Model is Secretly a Reward Model*. Singhal et al. (2023), *A Long Way to Go: Investigating Length Correlations in RLHF*.

**RL with a verifier and reasoning.** Shao et al. (2024), *DeepSeekMath* — GRPO. DeepSeek-AI (2025), *DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning*. Yu et al. (2025), *DAPO: An Open-Source LLM Reinforcement Learning System at Scale*. Liu et al. (2025), *Understanding R1-Zero-Like Training: A Critical Perspective* — Dr. GRPO. Yue et al. (2025), *Does Reinforcement Learning Really Incentivize Reasoning Capacity in LLMs Beyond the Base Model?* Chen et al. (2021), *Evaluating Large Language Models Trained on Code* — pass@k. Mukherjee et al. (2025), *Reinforcement Learning Finetunes Small Subnetworks in Large Language Models*. Qwen Team (2025), *Qwen3 Technical Report*.

**The cost of fine-tuning.** McCloskey & Cohen (1989), *Catastrophic Interference in Connectionist Networks*. Biderman et al. (2024), *LoRA Learns Less and Forgets Less*. Shenfeld, Pari & Agrawal (2025), *RL's Razor: Why Online Reinforcement Learning Forgets Less*. Gekhman et al. (2024), *Does Fine-Tuning LLMs on New Knowledge Encourage Hallucinations?* Qi et al. (2023), *Fine-tuning Aligned Language Models Compromises Safety, Even When Users Do Not Intend To!* Betley et al. (2025), *Emergent Misalignment: Narrow Finetuning Can Produce Broadly Misaligned LLMs*.